# How to Run This Notebook on Your Device  ?

This notebook uses files stored in a shared Google Drive folder.

Before running:

1. Open Google Drive.
2. Go to **Shared with me**.
3. Find the project folder:
   - `WaDaBa_Plastic_Classification_YOLO`
4. Right-click the folder.
5. Select **Organise → Add shortcut**.
6. Choose **My Drive**.
7. Open this notebook in Google Colab.
8. Run the cells from the top.

If Google Drive disconnects during execution, use:

Runtime → Disconnect and delete runtime

Then run again from the Drive mount step.

# Step 1: Plastic Waste Classification using YOLO (WaDaBa Dataset)

## Objective
The objective of this project is to develop a computer vision model capable of classifying different types of plastic waste into meaningful categories such as PET, PP, PS, PE-HD, and Others.

Unlike simple binary classification (plastic vs non-plastic), this project focuses on multi-class classification to improve real-world recycling applications.

## Dataset
Dataset used: WaDaBa Plastic Dataset  
- ~4000 images  
- Real-world waste images  
- Preprocessed and organised into structured classes  

## Approach
1. Data Cleaning & Organisation
2. Dataset Splitting (Train/Validation/Test)
3. Baseline Model (YOLO11n-cls)
4. Model Improvement (YOLO11s-cls + tuning)
5. Evaluation and Comparison

## Tools Used
- Python
- Ultralytics YOLO
- Google Colab (T4 GPU)

### Step 1.1 — WaDaBa Dataset Preparation Workflow

The original WaDaBa dataset was distributed as multiple compressed image folders grouped by collection batches rather than machine-learning-ready class folders.

Therefore, additional preprocessing was required before training the YOLO classification models.

The following preprocessing workflow was performed locally using Python:
1.   Extracted all WaDaBa archive folders
2.   Parsed plastic-type codes embedded inside image filenames
3.   Organised images into class-based folders:
*   PET
*   PE-HD
*   PP
*   PS
*   Other

4.   Removed unsupported or empty classes
5.   Split dataset into:
*   Training set (70%)
*   Validation set (20%)
*   Test set (10%)

6.   Exported final dataset in YOLO classification folder structure


This preprocessing improved dataset usability and ensured compatibility with the Ultralytics YOLO classification pipeline.

Dataset organisation and splitting scripts were implemented using Python automation scripts.

### Examples: of Dataset Organisation and Dataset Splitting

```
# Example: Extract plastic type from filename
# 0004_a01b05c2d0e1f0g1h1.jpg
# a01 = PET

plastic_map = {
    "01": "PET",
    "02": "PE-HD",
    "05": "PP",
    "06": "PS",
    "07": "Other"
}
```





```
TRAIN_RATIO = 0.70
VAL_RATIO = 0.20
TEST_RATIO = 0.10

```



# Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Step 3 — Install YOLO

In [ ]:
!pip install ultralytics

# Step 4 — Navigate to the project

In [ ]:
import os
from pathlib import Path

PROJECT_FOLDER_NAME = "WaDaBa_Plastic_Classification_YOLO"

SEARCH_ROOTS = [
    "/content/drive/MyDrive",
    "/content/drive/Shareddrives"
]

def find_project_folder(folder_name):
    for root in SEARCH_ROOTS:
        root_path = Path(root)
        if root_path.exists():
            matches = list(root_path.rglob(folder_name))
            if matches:
                return str(matches[0])
    raise FileNotFoundError(
        f"Project folder '{folder_name}' not found. "
        "Add the shared folder shortcut to My Drive first."
    )

PROJECT_PATH = find_project_folder(PROJECT_FOLDER_NAME)
os.chdir(PROJECT_PATH)

print("✅ Project folder found:", PROJECT_PATH)
print("✅ Current directory:", os.getcwd())

# Step 5 — Unzip dataset

In [ ]:
import os
import zipfile

dataset_zip = "04_yolo_cls_dataset.zip"
dataset_folder = "04_yolo_cls_dataset"

if os.path.exists(dataset_folder):
    print("✅ Dataset already exists. Skipping unzip.")
elif os.path.exists(dataset_zip):
    print("✅ Dataset zip found. Extracting...")
    with zipfile.ZipFile(dataset_zip, "r") as zip_ref:
        zip_ref.extractall(".")
    print("✅ Dataset extracted successfully.")
else:
    raise FileNotFoundError(
        "04_yolo_cls_dataset.zip not found. Put it inside WaDaBa_Plastic_Classification_YOLO folder."
    )

### Step 5.1 — Runtime Mode

In [ ]:
# Step 5.1 — Runtime mode

# False = load saved trained models from shared Google Drive
# True = retrain models from scratch
RETRAIN_MODE = False

# Keep this False for assessment marking.
# If True, the notebook may try to train again if saved weights are missing.
AUTO_TRAIN_IF_MISSING = False

print("RETRAIN_MODE:", RETRAIN_MODE)
print("AUTO_TRAIN_IF_MISSING:", AUTO_TRAIN_IF_MISSING)

To improve efficiency, the dataset was extracted once and reused for subsequent experiments.
Repeated extraction was avoided to reduce computational overhead and execution time in the Colab environment.

# Step 6 — Dataset Verification

In [ ]:
import os

base_path = "04_yolo_cls_dataset"

for split in ["train", "val", "test"]:
    print(f"\n{split.upper()} classes:")
    print(os.listdir(f"{base_path}/{split}"))

# Step 7 — Dataset distribution

In [ ]:
from collections import defaultdict
import os

dataset_path = "04_yolo_cls_dataset/train"

class_counts = {}

for cls in os.listdir(dataset_path):
    cls_path = os.path.join(dataset_path, cls)
    if os.path.isdir(cls_path):
        class_counts[cls] = len(os.listdir(cls_path))

print("\nTraining Data Distribution:")
for k, v in class_counts.items():
    print(f"{k}: {v}")

# Step 8 — Observation

The dataset is imbalanced, with PET having significantly more images compared to other classes.

Classes PVC and PE-LD were not included in training due to absence of images.

This imbalance may affect model performance and bias predictions toward dominant classes.

This reflects a real-world challenge in machine learning datasets.

### Step 8.1 Removing .DS_Store files created by Mac
System-generated files such as .DS_Store (created by macOS) were removed to prevent interference with dataset loading and model training.

In [ ]:
import os

removed = 0

for root, dirs, files in os.walk("04_yolo_cls_dataset"):
    for file in files:
        if file == ".DS_Store":
            file_path = os.path.join(root, file)
            os.remove(file_path)
            removed += 1

print(f"✅ Removed {removed} .DS_Store files")

# Step 9 — BASELINE MODEL TRAINING

In [ ]:
# Step 9 — Baseline Model Training / Loading.

from ultralytics import YOLO
from pathlib import Path
import os
import time

time.sleep(3)

DATASET_DIR = Path(PROJECT_PATH) / "04_yolo_cls_dataset"
RUNS_CLASSIFY_DIR = Path(PROJECT_PATH) / "runs" / "classify"

print("Dataset path:", DATASET_DIR)
print("Runs classify path:", RUNS_CLASSIFY_DIR)

RETRAIN_MODE = False
AUTO_TRAIN_IF_MISSING = False

# Find saved baseline model automatically
baseline_candidates = list(RUNS_CLASSIFY_DIR.glob("baseline_yolo11n_cls*/weights/best.pt"))

if baseline_candidates:
    baseline_model_path = baseline_candidates[0]
else:
    baseline_model_path = RUNS_CLASSIFY_DIR / "baseline_yolo11n_cls" / "weights" / "best.pt"

print("Baseline model path:", baseline_model_path)

if RETRAIN_MODE or (AUTO_TRAIN_IF_MISSING and not baseline_model_path.exists()):

    print("🔁 Training baseline model locally...")

    LOCAL_RUNS_DIR = Path("/content/runs/classify")
    LOCAL_RUNS_DIR.mkdir(parents=True, exist_ok=True)

    model = YOLO("yolo11n-cls.pt")

    results = model.train(
        data=str(DATASET_DIR),
        epochs=10,
        imgsz=224,
        batch=32,
        project=str(LOCAL_RUNS_DIR),
        name="baseline_yolo11n_cls",
        exist_ok=True
    )

else:

    if not baseline_model_path.exists():
        raise FileNotFoundError(
            f"Saved baseline model not found. Checked here: {baseline_model_path}"
        )

    model = YOLO(str(baseline_model_path))
    print("✅ Loaded saved baseline model:", baseline_model_path)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset path: /content/drive/MyDrive/WaDaBa_Plastic_Classification_YOLO/04_yolo_cls_dataset
Runs classify path: /content/drive/MyDrive/WaDaBa_Plastic_Classification_YOLO/runs/classify
Baseline model path: /content/drive/MyDrive/WaDaBa_Plastic_Classification_YOLO/runs/classify/baseline_yolo11n_cls-2/weights/best.pt
✅ Loaded saved baseline model: /content/drive/MyDrive/WaDaBa_Plastic_Classification_YOLO/runs/classify/baseline_yolo11n_cls-2/weights/best.pt


# Step 10 — Evaluate baseline

In [ ]:
# Step 10 — Evaluate baseline safely

from pathlib import Path

# Save validation outputs to local Colab storage, not shared Google Drive
LOCAL_VAL_DIR = Path("/content/runs/classify")

metrics = model.val(
    data=str(DATASET_DIR),
    project=str(LOCAL_VAL_DIR),
    name="baseline_val",
    exist_ok=True
)

print(metrics)

## Step 10.1 Visual Evaluation of Baseline Model

The training results and confusion matrix were generated automatically during model training.

### A. Training Performance (results.png)
The training curve shows a consistent decrease in loss and an increase in accuracy across epochs, indicating effective learning.

### B. Confusion Matrix (confusion_matrix.png)
The confusion matrix shows that most predictions are correctly classified, particularly for the PET class.

However, due to dataset imbalance, the model shows a tendency to favour dominant classes such as PET, which may lead to misclassification of minority classes.

In [ ]:
# Step 10.1 — Display baseline validation outputs safely

from IPython.display import Image, display
from pathlib import Path

# These files were generated by Step 10 in local Colab storage
baseline_results_path = Path("/content/runs/classify/baseline_val")

for image_name in ["confusion_matrix.png", "confusion_matrix_normalized.png"]:
    image_path = baseline_results_path / image_name

    if image_path.exists():
        print("Displaying:", image_path)
        display(Image(filename=str(image_path)))
    else:
        print("Missing image:", image_path)

# Step 11 — Predictions

In [ ]:
# Step 11 — Predictions safely using local Colab storage

from pathlib import Path
from IPython.display import Image, display

sample_dir = Path(PROJECT_PATH) / "sample_predictions"
local_predict_dir = Path("/content/runs/classify")

prediction_results = model.predict(
    source=str(sample_dir),
    save=True,
    project=str(local_predict_dir),
    name="baseline_predictions",
    exist_ok=True
)

pred_dir = local_predict_dir / "baseline_predictions"

print("✅ Predictions saved to:", pred_dir)

# Display first 8 prediction images only
image_files = list(pred_dir.glob("*.jpg")) + list(pred_dir.glob("*.png"))

for image_path in image_files[:8]:
    display(Image(filename=str(image_path)))

## Step 11.1 Baseline Model Results and Observation

The baseline YOLO11n-cls model was trained for 10 epochs using the WaDaBa plastic waste classification dataset.

Final baseline validation result:

- Top-1 Accuracy: 99.1%
- Top-5 Accuracy: 100%
- Final training loss: 0.1404

The loss decreased consistently across training epochs, showing that the model successfully learned useful visual features from the dataset.

However, the dataset is highly imbalanced. PET has significantly more images than other classes, while the Other class has only a small number of samples. Therefore, the high baseline accuracy should be interpreted carefully, because the model may be biased toward majority classes.

The prediction step was performed on a small sample of test images to visually confirm that the trained model can classify unseen plastic images.

## Step 11.2: Prediction Results and Analysis

A small subset of test images was used to evaluate model predictions.

Most images were correctly classified, particularly for the PET class, which aligns with the high Top-1 accuracy achieved during validation.

However, it was observed that the model tends to predict PET more frequently. This is likely due to dataset imbalance, where PET has significantly more training samples compared to other classes.

This indicates a bias toward the dominant class, which is a common issue in real-world machine learning datasets.

Overall, the model demonstrates strong classification performance, but further improvements are needed to handle minority classes more effectively.

# Step 12: Improved Model (YOLO11s)

To improve the baseline model, a larger YOLO classification model was used.

The baseline model used YOLO11n-cls, which is lightweight and fast. For improvement, YOLO11s-cls was selected because it has higher model capacity and can learn more detailed visual features.

The improved model was trained for 20 epochs to allow more learning time compared to the 10-epoch baseline.

In [ ]:
# Step 12 — Improved Model Training / Loading

from ultralytics import YOLO
from pathlib import Path
import os

DATASET_DIR = Path(PROJECT_PATH) / "04_yolo_cls_dataset"
RUNS_CLASSIFY_DIR = Path(PROJECT_PATH) / "runs" / "classify"
RUNS_CLASSIFY_DIR.mkdir(parents=True, exist_ok=True)

improved_model_path = RUNS_CLASSIFY_DIR / "improved_yolo11s_cls" / "weights" / "best.pt"

should_train_improved = RETRAIN_MODE or (AUTO_TRAIN_IF_MISSING and not improved_model_path.exists())

if should_train_improved:
    print("🔁 Training improved model...")

    model_improved = YOLO("yolo11s-cls.pt")

    results_improved = model_improved.train(
        data=str(DATASET_DIR),
        epochs=20,
        imgsz=224,
        batch=32,
        project=str(RUNS_CLASSIFY_DIR),
        name="improved_yolo11s_cls",
        exist_ok=True
    )

else:
    if not improved_model_path.exists():
        raise FileNotFoundError("Improved model not found. Set AUTO_TRAIN_IF_MISSING=True.")

    print("⚡ Loading saved improved model...")
    model_improved = YOLO(str(improved_model_path))

print("✅ Improved model ready:", improved_model_path)

# Step 13 — Evaluate Improved Model

In [ ]:
# Step 13 — Evaluate improved model safely

from pathlib import Path

LOCAL_VAL_DIR = Path("/content/runs/classify")

metrics_improved = model_improved.val(
    data=str(DATASET_DIR),
    project=str(LOCAL_VAL_DIR),
    name="improved_val",
    exist_ok=True
)

print(metrics_improved)

# Step 14 — Visual Evaluation of Improved Model


The improved YOLO11s-cls model was evaluated using the validation dataset.

The training graph and confusion matrix were generated automatically by YOLO.

The improved model achieved:

- Top-1 Accuracy: 99.5%
- Top-5 Accuracy: 100%

Compared with the baseline model, the improved model achieved slightly higher validation accuracy and used a larger model architecture with greater feature learning capacity.

## Step 14.1 Improved Graphs

In [ ]:
from IPython.display import Image, display
from pathlib import Path

improved_results_path = (
    Path(PROJECT_PATH)
    / "runs"
    / "classify"
    / "improved_yolo11s_cls"
)

display(Image(filename=str(improved_results_path / "results.png")))
display(Image(filename=str(improved_results_path / "confusion_matrix.png")))

# Step 15 — Baseline vs Improved Comparison


| Model | Epochs | Parameters | Top-1 Accuracy | Top-5 Accuracy |
|---|---:|---:|---:|---:|
| YOLO11n-cls Baseline | 10 | 1.53M | 99.1% | 100% |
| YOLO11s-cls Improved | 20 | 5.44M | 99.5% | 100% |

The improved YOLO11s-cls model achieved a Top-1 accuracy of 99.5%, compared with 99.1% from the baseline YOLO11n-cls model.

This is an improvement of 0.4 percentage points.

Although the accuracy improvement is small, the improved model has higher capacity and lower training loss, which indicates stronger feature learning.

However, the result must still be interpreted carefully because the dataset is imbalanced, with PET having significantly more images than other classes.

# Step 16 — Final Conclusion and Limitations

This project successfully developed a multi-class plastic waste classification model using the WaDaBa dataset and YOLO classification models.

The baseline YOLO11n-cls model achieved a Top-1 accuracy of 99.1%, while the improved YOLO11s-cls model achieved a Top-1 accuracy of 99.5%. This shows an improvement of 0.4 percentage points.

The improved model used a larger architecture and was trained for more epochs, allowing it to learn more detailed visual features from the dataset.

However, the dataset was imbalanced. PET had significantly more images than other classes, while PVC and PE-LD were not available in the extracted dataset. This means the model may perform better on majority classes and may not generalise equally across all plastic types.

Overall, the project demonstrates the full AI workflow: dataset preparation, preprocessing, training, evaluation, improvement, and result interpretation.

# Step 17 — Demo Web App

### Web App for testing Plastic types

You can also use below for the demo of this trained model usiong WaDaBa dataset or you can just wait to run this code below.

https://huggingface.co/spaces/lagnadlinus/plastic-waste-classification-yolo11

A simple prototype web application was created using Gradio to demonstrate how the trained model could be used in a real-world waste management system.

The prototype allows a user to upload an image of plastic waste and returns the predicted plastic type with confidence.

This concept could be extended for council waste management, where cameras installed near sorting belts could classify plastic types and support automated recycling workflows.

In [ ]:
# Installing gradio for making web app

!pip install gradio

#### Note:

Note: The Gradio demo may require third-party cookies to be enabled in Google Chrome when running inside Google Colab. If the embedded app does not appear, the model can still be tested using the prediction cells above. The Gradio app is included as a prototype interface, while the core model training, evaluation, and prediction pipeline runs independently of the web app.



In [ ]:
# Launch App - Enhanced Plastic Waste Classification Demo

import gradio as gr
from ultralytics import YOLO
from pathlib import Path

# Load best improved model
best_model_path = (
    Path(PROJECT_PATH)
    / "runs"
    / "classify"
    / "improved_yolo11s_cls"
    / "weights"
    / "best.pt"
)

if not best_model_path.exists():
    raise FileNotFoundError("Improved model not found. Run Step 12 first.")

app_model = YOLO(str(best_model_path))

plastic_info = {
    "PET": {
        "full_name": "Polyethylene Terephthalate",
        "uses": "Commonly used in drink bottles, food containers, and packaging.",
        "impact": "If not recycled, PET can remain in the environment for a very long time and contribute to plastic pollution.",
        "recycling": "Recycled PET can be used for new bottles, clothing fibres, packaging, and sometimes 3D printing filament."
    },
    "PE-HD": {
        "full_name": "High-Density Polyethylene",
        "uses": "Commonly used in milk bottles, detergent bottles, pipes, and hard plastic containers.",
        "impact": "It can build up in landfills and waterways if not collected and recycled properly.",
        "recycling": "Recycled PE-HD can be used for bins, pipes, outdoor furniture, and construction materials."
    },
    "PP": {
        "full_name": "Polypropylene",
        "uses": "Commonly used in food tubs, bottle caps, takeaway containers, and packaging.",
        "impact": "If not recycled, it can break into smaller plastic fragments and pollute soil and water.",
        "recycling": "Recycled PP can be used in automotive parts, storage boxes, and plastic products."
    },
    "PS": {
        "full_name": "Polystyrene",
        "uses": "Commonly used in foam packaging, disposable cups, trays, and insulation materials.",
        "impact": "Polystyrene is lightweight and can easily spread into the environment, harming wildlife and waterways.",
        "recycling": "Recycling is possible but often difficult due to contamination and low density."
    },
    "Other": {
        "full_name": "Other Plastic Type",
        "uses": "This category may include mixed or less common plastic materials.",
        "impact": "Mixed plastics are harder to recycle and may end up in landfill if not sorted correctly.",
        "recycling": "Future improvements would need more detailed sub-classification for better recycling decisions."
    }
}

def classify_plastic(image):
    results = app_model.predict(image)
    result = results[0]

    class_id = result.probs.top1
    confidence = result.probs.top1conf.item()
    class_name = result.names[class_id]

    info = plastic_info.get(class_name, {
        "full_name": "Unknown plastic type",
        "uses": "No information available.",
        "impact": "No information available.",
        "recycling": "No information available."
    })

    output_text = f"""
### Predicted Plastic Type: {class_name}

**Full name:** {info['full_name']}

**Confidence:** {confidence:.2%}

**Common uses:**
{info['uses']}

**Environmental impact if not recycled:**
{info['impact']}

**Possible recycling applications:**
{info['recycling']}

**Note:** This is a prototype trained on the WaDaBa dataset. Real-world accuracy may vary due to lighting, dirt, camera quality, and missing plastic classes.
"""

    return output_text

demo = gr.Interface(
    fn=classify_plastic,
    inputs=gr.Image(
        type="pil",
        label="Upload or capture plastic waste image",
        sources=["upload", "webcam"]
    ),
    outputs=gr.Markdown(label="Plastic Classification Result"),
    title="Plastic Waste Classification Demo",
    description="Upload or capture an image of plastic waste. The YOLO11s model predicts the plastic type and explains its uses, impact, and recycling potential."
)

demo.launch(
    share=True,
    debug=False,
    inline=True,
    prevent_thread_lock=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://26efb39f6717a10be6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#### Note:
If the Gradio app does not appear inside Colab, allow third-party cookies for colab.research.google.com or open the local URL shown in the cell output. The model code and prediction pipeline still run successfully.

# Step 18 — Web App Testing Observation

The Gradio web application was tested using external plastic images collected from Google image search.

The model performed correctly for some images, such as PET bottle images and some PS/polystyrene examples. However, it also produced incorrect predictions for several real-world images.

This happened because the model was trained only on the available WaDaBa classes: PET, PE-HD, PP, PS, and Other. PVC and PE-LD were not included because these classes had no available images in the extracted dataset.

Therefore, when PVC or PE-LD images were uploaded, the model could not predict those classes and instead assigned them to the closest known class such as PET or PS.

This shows an important limitation: high validation accuracy does not always mean perfect real-world performance. For real deployment in council waste management, the model would need a larger, more balanced dataset containing all plastic types and more real-world images from different environments.

# Step 19 — Conclusion

As a future real-world application, this type of AI-powered classification system could support Australian council waste management and recycling facilities by assisting automated sorting processes, reducing manual labour, and improving recycling accuracy. With larger and more balanced datasets, similar systems could potentially be integrated into conveyor belt camera systems, smart recycling bins, or robotic waste sorting infrastructure.

This notebook is designed to support two execution modes:

- `RETRAIN_MODE = False`: loads saved trained models and results for fast demonstration.
- `RETRAIN_MODE = True`: retrains the baseline and improved models from scratch using the available dataset.

This improves reproducibility because lecturers, markers, and team members can either review the saved outputs quickly or rerun the full training pipeline if required.

# References

Bobulski, J., & Piatkowski, J. (2018). PET waste classification method and Plastic Waste DataBase WaDaBa. Advances in Intelligent Systems and Computing, 681, 57–64.

Bobulski, J., & Kubanek, M. (2021). Deep Learning for Plastic Waste Classification System. Applied Computational Intelligence and Soft Computing, 2021. https://doi.org/10.1155/2021/6626948

WaDaBa Dataset. (n.d.). Plastic Waste DataBase of Images – WaDaBa. Retrieved from https://wadaba.pcz.pl/#home

Ultralytics. (n.d.). Ultralytics YOLO Documentation. https://docs.ultralytics.com/

Gradio. (n.d.). Gradio Documentation. https://www.gradio.app/

Google. (2025). NotebookLM [Large language model tool]. Retrieved from https://notebooklm.google/

OpenAI. (2026). ChatGPT (GPT-5.5) [Large language model]. Available at: https://chatgpt.com/
 (Accessed: 08 May 2026).